# 실험 05 — 암종 구별 유전자 가중치의 효과 검증

**연구 질문:** 학습 데이터에서 찾은 암종별 구별 유전자에 가중치를 높이면, 전체 Macro F1과 특정 암종 쌍의 오분류를 개선할 수 있는가?

- 실험 04 결과는 **가설 탐색**용이며 실험 05에서 유전자 목록이나 가중치를 선정하는 데 검증 데이터를 쓰지 않는다.
- 실험 05는 기존 Train 6,201명으로 **3-fold StratifiedGroupKFold**를 새로 구성한다. 프로파일이 중복된 환자는 같은 폴드에 둔다.
- 각 학습 폴드에서 암종별 변이율 차이가 큰 유전자를 추출하고 해당 폴드 검증 데이터에는 이미 정한 가중치만 적용한다.
- 하이퍼파라미터 `scale`은 미리 정한 1, 2, 4배를 비교한다. 결과가 좋아 보이는 한 분할만 골라 보고하지 않는다.
- 이번 분석에서 공식 Test 데이터는 **읽지 않는다.**

In [ ]:
# 필요 시 최초 1회 설치
# %pip install numpy pandas scipy scikit-learn matplotlib
from pathlib import Path
from collections import Counter
import hashlib, json
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from IPython.display import display

ROOT = Path.cwd()
OUT = ROOT / 'experiment_05'
OUT.mkdir(exist_ok=True)
DATA = ROOT / 'outputs'
assert (DATA/'X_train_binary.npz').exists(), 'outputs/X_train_binary.npz가 필요합니다.'
X = sparse.load_npz(DATA/'X_train_binary.npz').tocsr().astype(np.float32)
y = np.load(DATA/'y_train.npy',allow_pickle=True).astype(str)
with open(DATA/'gene_columns.json',encoding='utf-8') as f: genes = list(json.load(f))
assert X.shape == (len(y), len(genes))
print('Train:',X.shape, '암종:',len(np.unique(y)))

## 1. 동일 이진 프로파일 그룹 생성
동일한 유전자 0/1 패턴을 갖는 환자는 같은 그룹으로 묶는다. 중복 이진 프로파일이 검증 폴드와 학습 폴드에 동시에 등장하지 않도록 하기 위함이다.

In [ ]:
profiles=[]
for i in range(X.shape[0]):
    indexes = X.indices[X.indptr[i]:X.indptr[i+1]]
    profiles.append(hashlib.sha256(indexes.tobytes()).hexdigest())
groups = pd.factorize(profiles)[0]
print('고유 프로파일:',len(np.unique(groups)), '중복 프로파일 행 수:',len(groups)-len(np.unique(groups)))
cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
folds=list(cv.split(X,y,groups))
for i,(tr,va) in enumerate(folds,1):
    assert set(groups[tr]).isdisjoint(set(groups[va]))
    print(f'Fold {i}: train={len(tr)}, valid={len(va)}, 클래스={len(np.unique(y[va]))}')

## 2. 학습 폴드 내부에서만 유전자 가중치 계산

사전에 선정한 암종 쌍 `KIPAN–KIRC`, `GBMLGG–LGG`, `BRCA–OV`의 변이율 절대 차이가 큰 유전자를 **각 Fold 학습 부분에서만** 찾는다.

이는 *탐색적 선택*이므로 암종 쌍 자체를 완전히 독립적으로 검증한 성능이라고 해석할 수 없다. 후속 실험은 새로운 분할 또는 외부 데이터에서 재검증해야 한다.

In [ ]:
PAIRS=[('KIPAN','KIRC'),('GBMLGG','LGG'),('BRCA','OV')]
TOP_K=5
MIN_MUTATED=10

def weights_from_training(X_train,y_train,scale):
    weight=np.ones(X_train.shape[1],dtype=np.float32)
    rows=[]
    for a,b in PAIRS:
        Xa = X_train[y_train==a]
        Xb = X_train[y_train==b]
        if Xa.shape[0]==0 or Xb.shape[0]==0: continue
        total = np.asarray((Xa>0).sum(axis=0)+(Xb>0).sum(axis=0)).ravel()
        ra=np.asarray((Xa>0).mean(axis=0)).ravel()
        rb=np.asarray((Xb>0).mean(axis=0)).ravel()
        diff=np.abs(ra-rb)
        diff[total<MIN_MUTATED]=-1
        selected=np.argsort(-diff,kind='stable')[:TOP_K]
        selected=[j for j in selected if diff[j]>=0]
        for j in selected:
            weight[j]=max(weight[j],scale)
            rows.append({'pair':f'{a} ↔ {b}','gene':genes[j], 'rate_a':ra[j], 'rate_b':rb[j], 'abs_diff':diff[j]})
    return weight,pd.DataFrame(rows)

def model():
    return SGDClassifier(loss='log_loss',class_weight='balanced',max_iter=1500,tol=1e-3,random_state=42)


## 3. 3-Fold 학습과 평가
Baseline의 특징 가중치는 1배, 실험 모델은 2배와 4배다. 모델 구조와 나머지 학습 파라미터는 동일하다.

In [ ]:
metrics=[];pred_records=[];selection_records=[]
for fold,(tr,va) in enumerate(folds,1):
    Xtr,Xva=X[tr],X[va]
    for scale in (1,2,4):
        weight,selected=weights_from_training(Xtr,y[tr],scale)
        if len(selected):
            selected=selected.assign(fold=fold,scale=scale)
            selection_records.append(selected)
        m=model()
        m.fit(Xtr.multiply(weight).tocsr(),y[tr])
        pred=m.predict(Xva.multiply(weight).tocsr())
        metrics.append({'fold':fold,'weight_scale':scale, 'n_valid':len(va),
                        'macro_f1':f1_score(y[va],pred,average='macro',zero_division=0),
                        'accuracy':accuracy_score(y[va],pred)})
        for row_idx,truth,guess in zip(va,y[va],pred):
            pred_records.append({'fold':fold,'weight_scale':scale,'row_idx':int(row_idx),
                                 'true':truth,'pred':guess})
res=pd.DataFrame(metrics)
display(res)
display(res.groupby('weight_scale')[['macro_f1','accuracy']].agg(['mean','std']).round(4))

## 4. 혼동 암종 쌍 및 암종별 변화 평가
각 Fold의 참값을 기준으로 방향별 오분류 횟수와 비율을 구한다.

In [ ]:
p=pd.DataFrame(pred_records)
con=[]
for (fold,scale),g in p.groupby(['fold','weight_scale']):
    for a,b in PAIRS:
        for src,dst in ((a,b),(b,a)):
            actual=g[g['true']==src]
            con.append({'fold':fold,'weight_scale':scale,'true':src,'predicted':dst,
                        'support':len(actual),'errors':int((actual.pred==dst).sum()),
                        'error_rate':float((actual.pred==dst).mean()) if len(actual) else np.nan})
conf=pd.DataFrame(con)
print('방향별 평균 오분류율')
display(conf.groupby(['weight_scale','true','predicted'],as_index=False).error_rate.mean().round(4))
per=[]
for (fold,scale),g in p.groupby(['fold','weight_scale']):
    r=classification_report(g.true,g.pred,output_dict=True,zero_division=0)
    for c in sorted(np.unique(y)):
        per.append({'fold':fold,'weight_scale':scale,'class':c,
                    'support':r.get(c,{}).get('support',0),
                    'f1':r.get(c,{}).get('f1-score',0)})
per=pd.DataFrame(per)
display(per.groupby(['weight_scale','class']).f1.mean().unstack(0).round(3))

## 5. 결과 저장 및 해석 가이드

**주의:** 기존 실험 03의 검증 점수와 이번 3-Fold 평균 점수를 직접 비교해 개선 효과를 주장하면 안 된다. **이번 실험 내부에서** `scale=1` 기준선과 `scale=2/4`를 비교해야 한다. 특정 폴드에서 우연히 좋아진 결과는 반복 재검증이 필요하다.

가중치 변경은 예측에 영향을 주는 특징의 상대적 규모를 바꾸는 것이며, 생물학적 진단 근거를 제공하지 않는다.

In [ ]:
res.to_csv(OUT/'cv_model_comparison.csv',index=False,encoding='utf-8-sig')
p.to_csv(OUT/'cv_predictions.csv',index=False,encoding='utf-8-sig')
conf.to_csv(OUT/'pair_confusion_by_fold.csv',index=False,encoding='utf-8-sig')
per.to_csv(OUT/'class_f1_by_fold.csv',index=False,encoding='utf-8-sig')
if selection_records:
    pd.concat(selection_records,ignore_index=True).to_csv(OUT/'selected_genes_by_fold.csv',index=False,encoding='utf-8-sig')
summary=res.groupby('weight_scale').agg(macro_f1_mean=('macro_f1','mean'),
   macro_f1_std=('macro_f1','std'),accuracy_mean=('accuracy','mean')).reset_index()
summary.to_csv(OUT/'cv_summary.csv',index=False,encoding='utf-8-sig')
display(summary)
print('저장 경로:',OUT.resolve())